**MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

**Curso: TC5053 - Ciencia y analítica de datos**

Tecnológico de Monterrey

Prof Grettel Barceló Alonso

**Semana 3**
Bases, almacenes y manipulación de datos

---

*   NOMBRE: Juan Manuel Cuevas Gaytan
*   MATRÍCULA: A01197813


---

En esta actividad usarás una base de datos relacional basada en el informe de participación y la lista del top 10 de Netflix. Incluye películas y programas de televisión, así como información sobre temporadas, métricas de visualización, fechas de estreno, duración y más, organizada en las siguientes tablas:

* `movie`: Información general de las películas.

* `tv_show`: Información general de los programas de televisión.

* `season`: Datos de las temporadas asociadas a cada programa de TV.

* `view_summary`: Métricas de visualización y rendimiento de películas o temporadas.

Revisa con detalle su esquema para que comprendas cómo se relacionan las tablas anteriores.

**NOTA IMPORTANTE:** Asegúrate de responder *explícitamente* todos los cuestionamientos.


`PyMySQL` es una librería escrita en Python puro que funciona como conector (*driver*) para motores de bases de datos MySQL, permitiendo abrir conexiones, ejecutar consultas SQL y recuperar resultados directamente desde programas en Python.

In [ ]:
!pip install pymysql sqlalchemy firebase_admin

`SQLAlchemy` es una librería de Python que facilita la interacción con bases de datos y permite mantener un pool de conexiones eficiente, gestionar *commits* y *rollbacks* de forma automática y asegurar que múltiples conexiones simultáneas se manejen de manera segura, incluso cuando se ejecutan consultas SQL “puras”

In [1]:
# Importa las librerías necesarias
import pymysql
import sqlalchemy as sqla
import pandas as pd

Se crea una conexión (`conn`) para luego invocar declaraciones SQL.

In [2]:
db = sqla.create_engine('sqlite:///data/netflixdb.sqlite', pool_recycle=3600)
conn = db.connect()

Para que tus consultas sean más legibles y fáciles de mantener, puedes usar este formato multilínea con comillas triples y `sqla.text()`. Por ejemplo:

```
query = sqla.text("""
  SELECT ---
  FROM ---
  WHERE ---
""")
pd.read_sql(query, conn)
```

1.	Extrae toda la información de las películas que duran más de 5 horas.

In [3]:
query = """
	SELECT *
	FROM movie
	WHERE runtime>300;
"""

movies = pd.read_sql(query, conn)
movies

,available_globally,release_date,created_date,id,modified_date,runtime,locale,original_title,title
0,0,NaN,1704067200000,5793,1704067200000,3892,None,日本統一シリーズ: 映画シリーズ,Nihontouitsu Series: Film Series
1,0,NaN,1704067200000,5794,1704067200000,2120,None,釣りバカ日誌: 映画シリーズ,Free and Easy Series: Film Series
2,1,1.628226e+12,1704067200000,5874,1704067200000,312,None,NaN,Navarasa: Limited Series
3,0,NaN,1704067200000,9729,1704067200000,710,None,織田同志会 織田征仁: 映画シリーズ,Seiji Oda: Film Series
4,0,NaN,1704067200000,9730,1704067200000,427,None,キングダム～首領になった男～: 映画シリーズ,Kingdom ~ The Man Who Became the Top ~: Film S...


2.	¿Cuál es el porcentaje de películas disponibles únicamente en EU en relación con el total, excluyendo los valores `NULL`? La consulta SQL debe entregar directamente el porcentaje final. No se deben devolver resultados parciales para realizar el cálculo en Python.
	- **El 83.57 % de las películas está disponible únicamente en eu.**

In [4]:
query = """
	SELECT 100.0 * en_result.en_count / all_result.all_count as en_count
	FROM (
		SELECT count(*) en_count
		FROM movie
		WHERE available_globally = 0
	) as en_result,
	(
		SELECT count(*) all_count
		FROM movie
		WHERE available_globally IS NOT NULL
	) as all_result;
"""

en_count = pd.read_sql(query, conn)

en_count

,en_count
0,83.57175


3.	¿Cuáles son los idiomas o regiones originales en la tabla de películas?
	- **En la tabla solo existen dos valores: `en` (inglés) y NULL. Es decir, el único idioma registrado es el inglés; en el resto de las películas no se especifica el idioma original.**
* ¿Cuántos registros tienen el campo `locale` con valor `NULL`? (NULL en SQL ⇔ None en Python)
	- **11,474 registros tienen `locale` con valor NULL.**

In [5]:
query = """
	SELECT DISTINCT locale
	FROM movie;
"""

locale = pd.read_sql(query, conn)
locale

,locale
0,NaN
1,en


In [6]:
query = """
	SELECT count(*) null_values
	FROM movie
	WHERE locale IS NULL;
"""

locale_null = pd.read_sql(query, conn)
locale_null

,null_values
0,11474


4.	Asumiendo que los valores `NULL` en `locale` corresponden a otro idioma (diferente del inglés), el título original de la película NO debería coincidir con el título principal en dichos registros.
* Determina cuántas películas tienen títulos diferentes en estos dos campos (`title` y `original_title`).
	- **3,964 películas tienen títulos diferentes.**
*  ¿Coinciden los resultados (cantidad de `NULL` y títulos diferentes)? Si no es así, identifica qué características tienen los registros restantes.
	- **No coinciden, hay 11,474 registros con locale nulo, pero solo 3,964 tienen un título original distinto del título principal. Los 7,510 registros restantes tienen original nulo, y en SQL una comparación contra NULL no devuelve verdadero ni falso, por lo que no se incluyen en el resultado. Esa es la característica que comparten todos los registros faltantes.**
* Finalmente, concluye si la suposición de que los valores `NULL` en `locale` indican que la película está en otro idioma es válida.
	- **La suposición no se puede validar, aunque se cumple en los 3,964 registros con título distinto, más de la mitad de los registros con locale nulo tampoco tienen título original, por lo que no hay información suficiente para saber si están en otro idioma.**

In [7]:
query = """
	SELECT count(*) titulo_diferente_original
	FROM movie
	WHERE locale IS NULL AND original_title != title;
"""

titulo_diferente_original= pd.read_sql(query, conn)
titulo_diferente_original

,titulo_diferente_original
0,3964


In [8]:
query = """
	SELECT title, original_title, locale, available_globally
	FROM movie
	WHERE id NOT IN (
			SELECT id
			FROM movie
			WHERE locale IS NULL AND original_title != title
		) AND locale IS NULL;
"""

movies = pd.read_sql(query, conn)
movies.head()

,title,original_title,locale,available_globally
0,Damsel,None,None,1.0
1,Lift,None,None,1.0
2,The Super Mario Bros. Movie,None,None,0.0
3,The Boss Baby,None,None,0.0
4,Leo,None,None,1.0


In [9]:
query = """
  SELECT COUNT(*) null_origina_title
  FROM movie
  WHERE original_title IS NULL and locale IS NULL;
"""

pd.read_sql(query, conn)

,null_origina_title
0,7510


5.	Determina el título de la película que ha permanecido más tiempo en el top 10.

In [10]:
query = """
	SELECT title, cumulative_weeks_in_top10
	FROM
		movie
		INNER JOIN
		view_summary
		ON movie.id = view_summary.movie_id
	WHERE cumulative_weeks_in_top10 > 0
	ORDER BY cumulative_weeks_in_top10 DESC
	LIMIT 1;
"""

titles = pd.read_sql(query, conn)
titles.head()

,title,cumulative_weeks_in_top10
0,KPop Demon Hunters,65


6.	Identifica los 5 programas de TV con mayor cantidad de temporadas.

In [11]:
query = """
	SELECT tv_show.title, COUNT(DISTINCT(season_number)) seasons_count
	FROM
		tv_show
		INNER JOIN
		season
		ON tv_show.id = season.tv_show_id
	GROUP BY tv_show.id
	ORDER BY seasons_count DESC
	LIMIT 5;
"""

tv_shows = pd.read_sql(query, conn)
tv_shows.head()

,title,seasons_count
0,Grey's Anatomy,20
1,Naruto Shippuden,20
2,How do you like Wednesday?,20
3,Heartland (2007),17
4,It's Always Sunny in Philadelphia,16


7.	¿Cuáles son los intervalos de fechas de los resúmenes en la tabla `view_summary` de los períodos (`duration`) semestrales?
	- **2023-07-01 a 2023-12-31 y 2024-01-01 a 2024-06-30**

In [12]:
query = """
	SELECT DISTINCT
		date(start_date / 1000, 'unixepoch') AS start_date,
		date(end_date   / 1000, 'unixepoch') AS end_date
	FROM
		view_summary
	WHERE duration = 'SEMI_ANNUALLY'
	ORDER BY start_date;
"""
pd.read_sql(query, conn)


,start_date,end_date
0,2023-07-01,2023-12-31
1,2024-01-01,2024-06-30


8.	Ordena las temporadas de *Grey's Anatomy* según la cantidad de vistas registradas en el primer período semestral de 2024.
* ¿Cómo interpretarías los resultados obtenidos?
	- **Grey's Anatomy tiene la cantidad temporadas igual al maximo valor de numero de temporadas en el dataset. Las vistas disminuyen conforme avanzan las temporadas, esto puede deberse a dos causas, primero, en una serie tan larga solo los fans más constantes llegan a las últimas temporadas, a diferencia del espectador casual, y segundo, las vistas en las primeras temporadas podría indicar que la serie perdió calidad y mucha gente dejó de verla. Intenté consultar la fecha de estreno para comprobar si la caída de la temporada 19 a la 20 se debía a que la 20 se estrenó durante ese periodo, pero la fecha de estreno es nula en todas las temporadas, por lo que no fue posible verificarlo.**

In [13]:
query = """
	SELECT
		season.season_number, sum(views) sum_views
	FROM
		(
			SELECT
				id as tv_show_id, 
				title as tv_title
			FROM
				tv_show
			WHERE
				tv_title = 'Grey''s Anatomy'
		) tv
		INNER JOIN
		(
			SELECT 
				id as season_id, 
				season_number,
				tv_show_id
			FROM 
				season
		) season
		INNER JOIN
		(
			SELECT
				season_id,
				views,
				date(start_date / 1000, 'unixepoch') AS start_date
			FROM
				view_summary
			WHERE duration = 'SEMI_ANNUALLY'
				AND date(start_date / 1000, 'unixepoch') = '2024-01-01'
		) view_summary
		ON
			tv.tv_show_id = season.tv_show_id AND season.season_id = view_summary.season_id
	GROUP BY
		season.season_number
	ORDER BY sum_views DESC;
"""
pd.read_sql(query, conn)


,season_number,sum_views
0,1,3600000
1,2,3100000
2,5,2900000
3,4,2900000
4,3,2900000
5,6,2800000
6,7,2700000
7,8,2600000
8,9,2500000
9,10,2400000


In [14]:
# Queria ver si la fecha de estreno para comprar si las ultimas temporadas fueron las mas recientes 
# en relacion con el periodo, para saber si solo las personas que estaban mas al corriente fueron
# las que vieron solo las ultimas temporadas
query = """
	SELECT season_number, date(season.release_date / 1000, 'unixepoch') AS release_date
	FROM tv_show INNER JOIN season ON tv_show.id = season.tv_show_id
	WHERE tv_show.title = 'Grey''s Anatomy'
	GROUP BY season_number;
"""

pd.read_sql(query, conn)

,season_number,release_date
0,1,None
1,2,None
2,3,None
3,4,None
4,5,None
5,6,None
6,7,None
7,8,None
8,9,None
9,10,None


9.	Todas las consultas anteriores podrían hacerse también con la lógica relacional implementada en Pandas, que permite replicar la mayoría de las operaciones de SQL. Si los dataframes se han llenado como sigue, resuelve la consulta 8 utilizando las funciones de Pandas.

In [15]:
tv_show = pd.read_sql(sqla.text('SELECT * FROM tv_show'), conn)
season = pd.read_sql(sqla.text('SELECT * FROM season'), conn)
view_summary = pd.read_sql(sqla.text('SELECT * FROM view_summary'), conn)

In [16]:
netflix_df = pd.merge(tv_show, season, left_on='id', right_on='tv_show_id', suffixes=["_t", "_s"], how="inner")
netflix_df = pd.merge(netflix_df, view_summary, left_on='id_s', right_on='season_id', suffixes=["_s", "_v"], how="inner")

In [17]:
netflix_df[
    (netflix_df['title_t'] == 'Grey\'s Anatomy') &
    (pd.to_datetime(netflix_df['start_date'], unit='ms').dt.date == pd.to_datetime('2024-01-01').date()) &
    (netflix_df['duration'] == 'SEMI_ANNUALLY')
].groupby('season_number')['views'].sum().sort_values(ascending=False)

season_number
1.0     3600000
2.0     3100000
3.0     2900000
4.0     2900000
5.0     2900000
6.0     2800000
7.0     2700000
8.0     2600000
9.0     2500000
10.0    2400000
11.0    2200000
12.0    2000000
13.0    2000000
14.0    2000000
19.0    2000000
15.0    1900000
16.0    1800000
17.0    1700000
18.0    1700000
20.0     100000
Name: views, dtype: int64

`MySQL` es un sistema de gestión de bases de datos relacional, pero desde Python también es posible extraer información de bases de datos no relacionales, como `Firestore`, `MongoDB` o `Cassandra`, utilizando conectores o integraciones específicas. Esto permite combinar datos de diferentes fuentes según las necesidades del análisis o la aplicación.

En el siguiente ejercicio usarás un ejemplo con `Firestore` desde Python. Para ello utilizarás los módulos `credentials` y `firestore` de la biblioteca `firebase_admin`.

In [18]:
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

En `Firestore`, a diferencia de `MySQL` donde se utiliza un usuario y contraseña para conectarse, la autenticación se realiza mediante un archivo JSON que almacena las credenciales necesarias para acceder a la base de datos. Este archivo contiene las claves y la información de configuración que permiten a Python establecer la conexión de manera segura.

In [19]:
# consultancy.json almacena la clave privada para autenticar una cuenta y autorizar el acceso a los servicios
# A través de la función Certificate(), se regresa una credencial inicializada, que puedes utilizar para crear una nueva instancia de la aplicación
cred = credentials.Certificate('data/consultancy.json')
firebase_admin.initialize_app(cred)
db = firestore.client()

10.	Investiga cómo leer la colección `EMPLOYEE` y mostrar su contenido en un dataframe. Asegúrate de incluir el `id` en el resultado

In [20]:
employee = list(db.collection('EMPLOYEE').stream())
employee_data = [doc.to_dict() for doc in employee]
employee_df = pd.DataFrame(employee_data)
employee_df

,emp_lname,emp_fname,id,emp_hiredate
0,Senior,David,8LcLuxVHGAd2d9IQc5jF,1989-07-12 06:00:00+00:00
1,Smithfield,William,Fzd60D6Z2CU4n0wVV8YN,2004-06-04 05:00:00+00:00
2,News,John,lX5xuQ5w3i6ib2ExccWY,2000-11-08 06:00:00+00:00
3,Arbough,June,yocFj2lichOkbAj9NBfp,1996-12-01 06:00:00+00:00


In [21]:
firebase_admin.delete_app(firebase_admin.get_app())

---

**Referencias**

* Anthropic. (2026). Claude (Opus 5.5) [Modelo de lenguaje grande]. Utilizado para depuración de código, verificación de resultados y redaccion. https://claude.ai/

* McKinney, W. (2022). Python for Data Analysis (3rd ed.). O'Reilly Media.

* Pandas. (2026). Pandas Documentation[Documentación de software]. https://pandas.pydata.org/docs/index.html

* VanderPlas, J. (2022). Python Data Science Handbook (2nd ed.). O'Reilly Media.

---